# HeartMuLa 3B｜Kaggle T4 ×2｜官方原始權重

使用官方 **HeartMuLa-oss-3B-happy-new-year** 與 **HeartCodec-oss-20260123**，不使用量化權重或 ComfyUI。

| 元件 | 指定 GPU | 推論精度 |
|---|---|---|
| HeartMuLa 音樂生成 | GPU 0 | FP16 |
| HeartCodec 音訊解碼 | GPU 1 | FP32 |

依序執行 6 個程式格。保留 Kaggle 預裝的 PyTorch／TorchAO，請在 Kaggle 啟用 **GPU T4 ×2** 與 **Internet**。模型權重放 `/tmp/heartmula/ckpt`，MP3 放 `/kaggle/working`。

> 官方支援分別指定兩個模型的 GPU；這不是將同一模型拆層到兩張 GPU。原始權重約 22.4 GB，尚未在你的 Kaggle Session 實機驗證能完整生成。

## 1. 安裝套件

In [ ]:
# Import environment and installation utilities.
# 匯入環境設定與安裝工具。
import os
import subprocess
import sys
from pathlib import Path

# Prepare paths for source code, model weights, and outputs.
# 準備原始碼、模型權重與輸出的儲存路徑。
ROOT = Path('/kaggle/working')
REPO = ROOT / 'heartlib'
SCRATCH = Path('/tmp/heartmula')
CKPT = SCRATCH / 'ckpt'
CKPT.mkdir(parents=True, exist_ok=True)

# Store the Hugging Face cache outside Kaggle's 20 GB output directory.
# 將 Hugging Face 快取放在 Kaggle 20 GB 輸出目錄以外。
os.environ['HF_HOME'] = str(SCRATCH / 'huggingface')
os.environ['HF_XET_CACHE'] = str(SCRATCH / 'huggingface' / 'xet')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# Download the official source only when it is missing.
# 只有官方原始碼尚未存在時才下載。
if not (REPO / 'pyproject.toml').is_file():
    # Clone the official repository without verbose Git messages.
    # 安靜地取得官方程式庫。
    subprocess.run(['git', 'clone', '-q', 'https://github.com/HeartMuLa/heartlib.git', str(REPO)], check=True)

# Install the model-loading dependencies without replacing Kaggle's PyTorch.
# 安裝模型載入相依套件，不更換 Kaggle 原有的 PyTorch。
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.57.0', 'tokenizers==0.22.1',
    'huggingface-hub==0.36.2', 'vector-quantize-pytorch==1.27.15',
    'soundfile',
], check=True)

# Install the official TorchTune API while preserving Kaggle's compatible TorchAO build.
# 安裝官方 TorchTune API，保留 Kaggle 與 PyTorch 相容的 TorchAO 版本。
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
    'torchtune==0.4.0',
], check=True)

# Register the official HeartMuLa package without rewriting PyTorch dependencies.
# 註冊官方 HeartMuLa 套件，避免重新安裝 PyTorch。
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', str(REPO)], check=True)

# Check model-library imports before downloading more than 22 GB of weights.
# 下載超過 22 GB 的模型前，先確認主要程式庫可以匯入。
subprocess.run([
    sys.executable, '-c',
    'import torchao, torchtune, torchaudio; from heartlib import HeartMuLaGenPipeline',
], check=True)

# Verify that Kaggle has supplied two CUDA GPUs.
# 確認 Kaggle 提供兩張 CUDA GPU。
import torch
assert torch.cuda.device_count() == 2, '請在 Kaggle Settings 選擇 GPU T4 ×2。'

# Show the GPU models available for component assignment.
# 顯示可分配模型元件的 GPU 名稱。
for index in range(2):
    # Print the name of the current GPU.
    # 顯示目前 GPU 的名稱。
    print(f'GPU {index}: {torch.cuda.get_device_name(index)}')

## 2. 設定歌詞、曲風與長度

In [ ]:
# Define original lyrics for music generation.
# 設定音樂生成使用的原創歌詞。
LYRICS = """
[Verse]
雨後的街道慢慢亮起燈
微風吹過熟悉的旅程

[Chorus]
讓我們迎著光向前走
把每個故事留在心中
""".strip()

# Describe the musical style using comma-separated tags.
# 使用逗號分隔的標籤描述音樂風格。
TAGS = 'soft,sad,ballad,longing'

# Set the maximum generated audio duration in seconds.
# 設定生成音訊的最長秒數。
DURATION = 20

# Save the lyrics as a UTF-8 text file.
# 將歌詞儲存為 UTF-8 文字檔。
LYRICS_FILE = ROOT / 'heartmula_lyrics.txt'
LYRICS_FILE.write_text(LYRICS, encoding='utf-8')

# Save the musical style tags as a text file.
# 將曲風標籤儲存為文字檔。
TAGS_FILE = ROOT / 'heartmula_tags.txt'
TAGS_FILE.write_text(TAGS, encoding='utf-8')

## 3. 下載官方模型

In [ ]:
# Import the official Hugging Face snapshot downloader.
# 匯入 Hugging Face 官方模型下載工具。
from huggingface_hub import snapshot_download

# Download the official generation configuration and tokenizer.
# 下載官方生成設定與 tokenizer。
snapshot_download(
    repo_id='HeartMuLa/HeartMuLaGen',
    local_dir=str(CKPT),
    allow_patterns=['tokenizer.json', 'gen_config.json'],
    max_workers=2,
)

# Download the original full-precision HeartMuLa 3B checkpoints.
# 下載官方原始精度的 HeartMuLa 3B 權重。
snapshot_download(
    repo_id='HeartMuLa/HeartMuLa-oss-3B-happy-new-year',
    local_dir=str(CKPT / 'HeartMuLa-oss-3B'),
    allow_patterns=['config.json', '*.safetensors', '*.safetensors.index.json'],
    max_workers=2,
)

# Download the matching original full-precision HeartCodec checkpoints.
# 下載配對的官方原始精度 HeartCodec 權重。
snapshot_download(
    repo_id='HeartMuLa/HeartCodec-oss-20260123',
    local_dir=str(CKPT / 'HeartCodec-oss'),
    allow_patterns=['config.json', '*.safetensors', '*.safetensors.index.json'],
    max_workers=2,
)

# Show where the official weights were stored.
# 顯示官方模型權重的儲存位置。
print('Model directory:', CKPT)

## 4. TorchTune RoPE 相容修正

In [ ]:
# Locate the cache initialization code in the official package.
# 找到官方套件內的 cache 初始化程式碼。
model_file = REPO / 'src/heartlib/heartmula/modeling_heartmula.py'

# Read the current implementation before patching.
# 在修改前先讀取目前的程式內容。
source = model_file.read_text(encoding='utf-8')

# Match the cache setup that may require RoPE initialization.
# 找出可能需要先初始化 RoPE 的 cache 設定。
original = '        with device:\n            self.backbone.setup_caches(max_batch_size, dtype)'

# Prepare a compatibility adjustment without modifying model weights.
# 準備不更動模型權重的相容性修正。
replacement = '''        with device:
            for module in self.modules():
                if hasattr(module, 'rope_init') and getattr(module, 'is_cache_built', True) is False:
                    module.rope_init()
            self.backbone.setup_caches(max_batch_size, dtype)'''

# Apply the patch only to the known source layout.
# 僅在已知原始碼格式下套用修正。
if 'module.rope_init()' not in source:
    # Require the expected cache setup to avoid modifying unrelated code.
    # 確認預期的 cache 初始化程式存在，避免誤改其它程式。
    if original not in source:
        # Explain why the current official source cannot be patched safely.
        # 說明目前官方程式碼無法安全套用修正的原因。
        raise RuntimeError('HeartMuLa 的 RoPE cache 初始化程式已變更，需要重新檢查版本。')
    # Save the compatible cache initialization to the source file.
    # 將相容的 cache 初始化方式寫回原始碼。
    model_file.write_text(source.replace(original, replacement, 1), encoding='utf-8')

## 5. 使用 GPU 0 + GPU 1 生成 MP3

In [ ]:
# Import the tools used to launch and time the official generator.
# 匯入執行官方生成器與計時的工具。
import time

# Define the final MP3 path under Kaggle's output directory.
# 將最後的 MP3 放在 Kaggle 輸出目錄。
output = ROOT / 'heartmula_output.mp3'

# Remove the previous output to avoid showing an outdated file after failure.
# 移除舊輸出，避免執行失敗時誤用先前的檔案。
output.unlink(missing_ok=True)

# Run the original HeartMuLa model on GPU 0 and the original HeartCodec on GPU 1.
# 在 GPU 0 執行原始 HeartMuLa，並於 GPU 1 執行原始 HeartCodec。
command = [
    sys.executable, '-u', str(REPO / 'examples/run_music_generation.py'),
    '--model_path', str(CKPT), '--version', '3B',
    '--lyrics', str(LYRICS_FILE), '--tags', str(TAGS_FILE),
    '--save_path', str(output),
    '--max_audio_length_ms', str(DURATION * 1000),
    '--mula_device', 'cuda:0', '--codec_device', 'cuda:1',
    '--mula_dtype', 'float16', '--codec_dtype', 'float32',
    '--lazy_load', 'false',
]

# Start the timer before invoking the generation process.
# 在執行生成程序前開始計時。
start_time = time.perf_counter()

# Execute the official inference program and stop on failure.
# 執行官方推論程式，發生錯誤時立即停止。
subprocess.run(command, cwd=REPO, check=True)

# Record total inference and MP3-saving time.
# 計算推論與 MP3 儲存的總耗時。
elapsed = time.perf_counter() - start_time

# Confirm that a nonempty MP3 was created.
# 確認 MP3 已成功產生且不是空檔案。
assert output.is_file() and output.stat().st_size > 1024, '找不到有效的 MP3 輸出。'

# Show the completed file and elapsed time.
# 顯示完成的檔案及生成耗時。
print(f'MP3: {output}')
print(f'生成耗時：{elapsed / 60:.2f} 分鐘（{elapsed:.1f} 秒）')

## 6. 試聽與下載

In [ ]:
# Import Jupyter audio playback and file links.
# 匯入 Jupyter 音訊播放與檔案連結工具。
from IPython.display import Audio, FileLink, display

# Preview the generated MP3 in Kaggle.
# 在 Kaggle 中試聽生成的 MP3。
display(Audio(filename=str(output)))

# Provide a download link for the MP3.
# 提供 MP3 的下載連結。
display(FileLink(str(output)))